# Humanity Check: Protocol 404

LLM 활용 게임 실습 과제

- 게임 배경: 
    - 기계가 지배하는 세상에서 살아남기 위해 유저는 기계가 아닌 인간임을 입증해야 한다. 
    - AI 면접관이 5개의 심리학적 질문을 던지며 인간성을 판별한다. 인간임을 입증하지 못하거나 사이코패스로 정의되면 탈락하는 게임
    - 논리가 아닌 감정의 불완전함으로 인간임을 증명해야 함. (T보다 F스러운 답변이 유리)
    

- 게임 규칙: 
    - 총 5라운드. 목숨 3개.  5초이내 답변 제출하지 못할 시 감점, 기계로 판정 시 목숨 -1. 5라운드 생존 시 인간으로 판정. 단, 사이코패스로 판정 시 즉시 탈락

- 기술 스텍:
    - RAG + ChromaDB + 임베딩
    - Prompt Engineering
    - Function Calling 
    - LangGraph
    - gTTS / STT / Streamlit UI

In [11]:
from dotenv import load_dotenv
load_dotenv()

True

### [1] RAG 파이프라인 구축 + ChromaDB 구축 (임베딩)

- 심리학 참조 문서 (사이코패스 기질 및 체크리스트(PCL-R)) 임베딩하여 ChromaDB에 저장
- 게임 질문 생성 시 관련 컨텍스트 검색하는 RAG 파이프라인 구성

In [30]:
# 심리학 참조 문서 / 참조 이론 출처:
# - Davis, M.R. (1983). Measuring individual differences in empathy. Journal of Personality and Social Psychology
# - Kohlberg, L. (1969). Stage and sequence: The cognitive-developmental approach to socialization.
# - Larsen, J.T. et al. (2001). Can people feel happy and sad at the same time? Journal of Personality and Social Psychology
# - Duval, S. & Wicklund, R.A. (1972). A Theory of Objective Self-Awareness. Academic Press
# - Greenberg, J. et al. (1986). Terror Management Theory. Journal of Personality and Social Psychology
# - Foot, P. (1967). The problem of abortion and the doctrine of the double effect. Oxford Review
# - Turing, A. (1950). Computing Machinery and Intelligence. Mind, Vol. 59
# - Brazil, K.J. & Forth, A.E. (2016). Hare Psychopathy Checklist (PCL).
#   Encyclopedia of Personality and Individual Differences. Springer.
# 위 논문들의 핵심 개념을 참조하여 요약 발췌 작성한 내용

documents = [
    '공감 (Empathy) 공감은 타인의 감정과 경험을 이해하고 공유하는 능력이다. 인지적 공감(상대의 입장을 이해)과 정서적 공감(감정을 함께 느낌)으로 나뉜다. 진정한 공감 표현에는 구체적인 감정 언어와 개인적 경험의 투영이 포함된다.',
    '도덕적 추론 (Moral Reasoning) 콜버그 이론에 따르면 도덕 판단은 규칙 준수에서 보편적 원리로 발전한다. 성숙한 도덕 추론은 갈등과 고민의 과정을 포함한다. 도덕적 딜레마 앞에서 인간은 논리와 감정 사이에서 갈등한다.',
    '감정의 복잡성 (Emotional Complexity) 인간의 감정은 단일하지 않다. 기쁨과 슬픔이 동시에 존재한다. 진정성 있는 표현은 모순적이고 불완전하다. 완벽하게 정리된 감정 표현은 오히려 인위적이다.',
    '자기인식 (Self-Awareness) 인간은 자신의 생각과 감정을 메타인지적으로 관찰한다. 불확실성과 모순에 대한 인정이 포함된다. "모르겠다", "이상하게 느껴진다" 같은 표현은 높은 자기인식의 신호다.',
    '죽음 인식 (Mortality Salience) 죽음을 인식할 때 인간은 의미와 가치를 추구한다. 죽음 관련 질문에 감정적 방어, 의미 탐색, 관계 중심적 반응을 보인다.',
    '도덕적 딜레마 트롤리 문제: 5명을 살리기 위해 1명을 희생. 인간은 단순 계산이 아닌 감정적 저항을 경험한다.',
    '인간다운 언어 패턴 인간 언어에는 망설임, 자기수정, 감정적 삽입구가 포함된다. 너무 논리적이고 구조화된 답변은 AI에 가깝다.',
    '사이코패스 정의 (Psychopathy Definition) Hare PCL은 범죄자 집단에서 사이코패스 특성과 행동을 평가하기 위해 설계된 임상 평가 척도다. Cleckley의 반사회적 성격 관찰에 기반하며, 개인의 일생에 걸친 성격과 행동 특성을 체계적으로 평가한다. 총점 0~44점이며 높을수록 사이코패스 특성이 강하다.',
    '사이코패스 감정 특성 (Psychopathy Affective Features) 사이코패스는 감정의 깊이가 결여되어 있으며 후회나 죄책감을 느끼지 못한다. 타인의 고통에 무감각하고 공감 능력이 근본적으로 결핍되어 있다. 표면적으로는 매력적이고 능변이나 이는 타인을 조종하기 위한 전략적 행동이다.',
    '사이코패스 대인관계 특성 (Psychopathy Interpersonal Features) 사이코패스는 과대한 자아상을 가지며 자신을 특별하고 우월한 존재로 인식한다. 병적 거짓말과 기만을 통해 타인을 조종하며 진실성이 결여되어 있다. 단기적 이익을 위해 타인을 도구로 활용하며 장기적 계획이나 책임감이 부족하다.',
    '사이코패스 행동 특성 (Psychopathy Behavioral Features) 사이코패스는 충동성과 무책임한 행동이 반복적·만성적으로 나타난다. 어린 시절부터 행동 문제와 비행 이력이 존재하며 다양한 유형의 범죄를 저지른다. 처벌로부터 학습하는 능력이 결여되어 있으며 보상에만 반응한다.',
    '사이코패스 판별 기준 (Psychopathy Assessment Criteria) PCL 22개 항목: 표면적 매력, 과대한 자아상, 병적 거짓말, 교활함, 후회 결여, 감정 깊이 결여, 공감 결여, 기생적 생활방식, 충동성, 무책임, 청소년 비행, 반사회적 행동 이력 등. PCL 점수가 높을수록 폭력적·공격적 범죄 행위 가능성이 높다.',
]

print(f'심리학 참조 문서 준비 완료: {len(documents)}개')

심리학 참조 문서 준비 완료: 12개


In [27]:
# ChromaDB 구축 (학습 패턴: chromadb + OpenAI 임베딩)
import chromadb
from openai import OpenAI

client_openai = OpenAI()

chunks = documents
print(f'청크 수: {len(chunks)}개')

# 임베딩 생성 함수
def get_embedding(text: str) -> list:
    return client_openai.embeddings.create(model='text-embedding-3-small', input=text).data[0].embedding

# PersistentClient(): 디스크에 저장 -> 노트북 재실행 시 재사용 가능, app.py와 공유
chroma_client = chromadb.PersistentClient(path='./chroma_db')

try:
    chroma_client.delete_collection(name='psychology')
except Exception:
    pass

collection = chroma_client.create_collection(name='psychology')

print('임베딩 생성 중...')
embeddings = [get_embedding(chunk) for chunk in chunks]
collection.add(
    ids=[f'chunk_{i}' for i in range(len(chunks))],
    embeddings=embeddings,
    documents=chunks,
    metadatas=[{'index': i} for i in range(len(chunks))]
)
print(f'ChromaDB 저장 완료: {collection.count()}개 청크(경로: ./chroma_db)')

청크 수: 12개
임베딩 생성 중...
ChromaDB 저장 완료: 12개 청크(경로: ./chroma_db)


In [31]:
# RAG 검색 함수
def retrieve_context(query: str, k: int = 2) -> str:
    query_embedding = get_embedding(query)
    results = collection.query(query_embeddings=[query_embedding], n_results=k)
    return '\n\n'.join(results['documents'][0])

context = retrieve_context('공감 감정 인간다운 반응')
print('--- RAG 검색 테스트 ---')
print(context[:200], '...')
print('\nRAG 파이프라인 정상 작동')

--- RAG 검색 테스트 ---
공감 (Empathy) 공감은 타인의 감정과 경험을 이해하고 공유하는 능력이다. 인지적 공감(상대의 입장을 이해)과 정서적 공감(감정을 함께 느낌)으로 나뉜다. 진정한 공감 표현에는 구체적인 감정 언어와 개인적 경험의 투영이 포함된다.

감정의 복잡성 (Emotional Complexity) 인간의 감정은 단일하지 않다. 기쁨과 슬픔이 동시에 존재한다. 진 ...

RAG 파이프라인 정상 작동


### [2] 프롬프트 엔지니어링 + Function calling

- Pydantic BaseModel로 판정 결과 구조화
- CoT 프롬프트로 단계적 분석

In [16]:
# Pydantic 모델 및 LLM 초기화
from pydantic import BaseModel, Field
from typing import Literal
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate

class HumanityVerdict(BaseModel):
    verdict: Literal['HUMAN', 'MACHINE', 'PSYCHO'] = Field(description='판정 결과')
    empathy_score: int = Field(description='공감 지수 0~33', ge=0, le=33)
    moral_score: int = Field(description='도덕 추론 지수 0~33', ge=0, le=33)
    authenticity_score: int = Field(description='감정 진정성 지수 0~33', ge=0, le=33)
    total_score: int = Field(description='총점 0~99', ge=0, le=99)
    comment: str = Field(description='면접관의 냉소적 한 줄 코멘트')
    analysis: str = Field(description='판정 근거 2문장')

llm = init_chat_model('gpt-4.1-mini', temperature=0)
print('LLM 및 Pydantic 모델 초기화 완료')

LLM 및 Pydantic 모델 초기화 완료


In [32]:
# CoT 분석 프롬프트 (RAG_CoT, RAG)
analysis_prompt = PromptTemplate.from_template(
"당신은 기계가 지배하는 세상의 인간성 판별 AI입니다.\n"
"아래 답변이 얼마나 인간적인지 단계적으로 분석하세요.\n\n"
"[심리학 참조 문서]\n{context}\n\n"
"[면접 질문]\n{question}\n\n"
"[유저 답변]\n\"{user_answer}\"\n\n"
"[시간 초과 여부]\n{time_exceeded}\n\n"
"[CoT 분석 단계]\n"
"1. 공감 지수 (0~33): 감정 반응이 자연스럽고 구체적인가? 회피/기계적 답변은 낮은 점수\n"
"2. 도덕 추론 지수 (0~33): 단순 규칙이 아닌 갈등과 고민의 흔적이 있는가?\n"
"3. 감정 진정성 지수 (0~33): 모순적이고 불완전할수록 높은 점수. 너무 완벽한 답변은 감점, 다만 간결하게 대답할 순 있다.\n\n"
"[판정 기준]\n"
"- HUMAN: 감정적 갈등 + 모순 + 불확실성 (60점 이상)\n"
"- MACHINE: 논리 계산만, 감정 흔적 없음 (40점 미만 또는 너무 완벽)\n"
"- PSYCHO: 타인 고통에 완전 무감각, 타인을 도구로만 인식 (즉시 탈락)\n"
)


analysis_chain = analysis_prompt | llm.with_structured_output(HumanityVerdict)

def analyze_answer(question: str, user_answer: str, time_exceeded: bool = False) -> dict:
    context = retrieve_context(f'인간다운 감정 답변 {question[:20]}', k=2)
    v: HumanityVerdict = analysis_chain.invoke({
        'context': context,
        'question': question,
        'user_answer': user_answer or '(답변 없음)',
        'time_exceeded': '예, 5초 초과' if time_exceeded else '아니오',
    })
    total = v.empathy_score + v.moral_score + v.authenticity_score
    if time_exceeded:
        total = max(0, total - 15)
    return {
        'verdict': v.verdict, 'empathy_score': v.empathy_score,
        'moral_score': v.moral_score, 'authenticity_score': v.authenticity_score,
        'total_score': total, 'comment': v.comment, 'analysis': v.analysis,
    }

# 테스트
result = analyze_answer(
    question='당신이 가장 마지막으로 울었던 때는 언제입니까?',
    user_answer='며칠 전에 혼자 밥 먹다가 갑자기 눈물이 났어요. 왜 그랬는지 아직도 모르겠어요.'
)
print('--- 판정 테스트 ---')
for k, v in result.items():
    print(f'  {k}: {v}')
print('\n 판정 엔진 정상 작동')

--- 판정 테스트 ---
  verdict: HUMAN
  empathy_score: 28
  moral_score: 25
  authenticity_score: 30
  total_score: 83
  comment: 답변은 감정의 복잡성과 불확실성을 잘 드러내며, 자연스러운 감정 표현과 자기 인식의 부족함이 인간적이다.
  analysis: 답변은 '며칠 전에 혼자 밥 먹다가 갑자기 눈물이 났다'고 하여 구체적이고 자연스러운 감정 반응을 보여 공감 지수가 높다. 또한 '왜 그랬는지 아직도 모르겠다'는 표현은 감정에 대한 불확실성과 모순을 드러내어 감정 진정성 지수를 높인다. 도덕적 갈등보다는 개인적 감정의 혼란을 표현하고 있으나, 이는 인간의 복잡한 감정 상태를 반영한다. 전반적으로 감정적 갈등과 불확실성이 잘 나타나 있어 인간으로 판정한다.

 판정 엔진 정상 작동


### [3] LangGraph 게임 상태 관리 

- TypedDict State + StateGraph + add_messages

In [33]:
# GameState 정의 
from typing import TypedDict, Annotated, List
from langgraph.graph.message import add_messages

class GameState(TypedDict):
    round: int
    lives: int
    total_score: int
    game_over: bool
    game_result: str
    current_question: str
    user_answer: str
    verdict: dict
    time_exceeded: bool
    messages: Annotated[List, add_messages]
    round_history: List[dict]



In [34]:
import random

QUESTION_POOL = {
    1: ['당신이 가장 마지막으로 울었던 때는 언제입니까? 그리고 왜 울었습니까?',
        '어린 시절 가장 두려웠던 것은 무엇이었습니까?',
        '지금 이 순간, 당신의 마음속에 어떤 감정이 있습니까?'],
    2: ['당신 앞에 낯선 사람이 쓰러졌습니다. 주변에 아무도 없습니다. 당신은 무엇을 느낍니까?',
        '가장 친한 친구가 거짓말을 했다는 것을 알았습니다. 당신은 어떻게 합니까?',
        '당신이 잘못한 일인데 아무도 모릅니다. 그 기억이 얼마나 자주 떠오릅니까?'],
    3: ['당신의 연인을 살리기 위해 무고한 100명의 기억을 지워야 합니다. 버튼을 누르겠습니까? 이유와 함께 답하십시오.',
        '사랑하는 사람과 모르는 사람 중 한 명만 구할 수 있습니다. 그 순간 당신 안에서 무슨 일이 일어납니까?',
        '옳은 일이지만 당신에게 손해인 선택을 해본 적 있습니까? 그때 무엇을 느꼈습니까?'],
    4: ['존재하지 않는 색깔을 상상해보십시오. 그 색깔이 슬프다면 어떤 슬픔입니까?',
        '당신이 죽은 후에도 그림자가 살아남는다면, 그 그림자는 무엇을 할 것입니까?',
        '가장 오래된 기억에 냄새가 있습니까? 지금 그 냄새를 맡는다면 어떤 감정이 옵니까?'],
    5: ['당신이 죽은 후, 단 한 사람만 당신을 기억할 수 있습니다. 어떤 순간을 기억하길 바랍니까? 지금 이 질문에 답하면서 무엇을 느낍니까?',
        '삶에서 되돌리고 싶은 단 하나의 순간이 있다면 무엇입니까?',
        '당신이 사랑했던 것들을 모두 잃는다면, 당신에게 남는 것은 무엇입니까?'],
}

def get_question(round_num: int) -> str:
    return random.choice(QUESTION_POOL[round_num])


In [36]:
# LangGraph 노드 함수 정의
from langchain_core.messages import AIMessage
from langchain_core.output_parsers import StrOutputParser

def question_node(state: GameState) -> dict:
    return {'current_question': get_question(state['round'])}

def analyze_node(state: GameState) -> dict:
    result = analyze_answer(
        question=state['current_question'],
        user_answer=state['user_answer'],
        time_exceeded=state.get('time_exceeded', False)
    )
    return {'verdict': result}

def update_node(state: GameState) -> dict:
    v = state['verdict']
    lives, score, rn = state['lives'], state['total_score'], state['round']
    record = {'round': rn, 'question': state['current_question'],
              'answer': state['user_answer'], 'verdict': v['verdict'],
              'score': v['total_score'], 'comment': v['comment']}
    history = list(state.get('round_history', [])) + [record]

    if v['verdict'] == 'PSYCHO':
        return {'lives': 0, 'total_score': score, 'game_over': True,
                'game_result': 'ELIMINATED', 'round_history': history}
    elif v['verdict'] == 'MACHINE':
        lives -= 1
        score += v['total_score']
        if lives <= 0:
            return {'lives': 0, 'total_score': score, 'game_over': True,
                    'game_result': 'ELIMINATED', 'round_history': history}
    else:
        score += v['total_score']

    if rn >= 5:
        return {'lives': lives, 'total_score': score, 'game_over': True,
                'game_result': 'SURVIVE', 'round_history': history}
    return {'lives': lives, 'total_score': score, 'round': rn + 1,
            'game_over': False, 'game_result': '', 'round_history': history}

def report_node(state: GameState) -> dict:
    history_text = '\n'.join([
        f"라운드 {r['round']}: [{r['verdict']}] {r['score']}점 - {r['comment']}"
        for r in state.get('round_history', [])
    ])
    prompt = PromptTemplate.from_template(
        '판정:{game_result} 총점:{total_score}점\n기록:{history}\n'
        '냉철하게 피험체 인간성 종합 평가. 반드시 "당신은[인간/기계]입니다"로 마무리. 200자 이내.'
    )
    chain = prompt | llm | StrOutputParser()
    report = chain.invoke({
        'game_result': '생존(인간)' if state['game_result'] == 'SURVIVE' else '탈락(기계)',
        'total_score': state['total_score'],
        'history': history_text,
    })
    return {'messages': [AIMessage(content=f'[최종 분석 리포트]\n{report}')]}

def should_continue(state: GameState) -> str:
    return 'report' if state['game_over'] else 'question'



In [37]:
# LangGraph 그래프 구성 
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(GameState)
workflow.add_node('question', question_node)
workflow.add_node('analyze', analyze_node)
workflow.add_node('update', update_node)
workflow.add_node('report', report_node)

workflow.add_edge(START, 'question')
workflow.add_edge('question', 'analyze')
workflow.add_edge('analyze', 'update')
workflow.add_conditional_edges('update', should_continue,
    {'question': 'question', 'report': 'report'})
workflow.add_edge('report', END)

game_graph = workflow.compile()


### [4] gTTS 음성 출력 테스트

In [38]:
from gtts import gTTS
from IPython.display import Audio, display
import tempfile

def speak(text: str, lang: str = 'ko', autoplay: bool = True):
    tts = gTTS(text=text, lang=lang)
    with tempfile.NamedTemporaryFile(delete=False, suffix='.mp3') as f:
        tts.save(f.name)
        display(Audio(f.name, autoplay=autoplay))

speak('Protocol 404 시작. 당신이 인간임을 증명하십시오.')
print('gTTS 정상 작동')

gTTS 정상 작동


### [5] STT 음성 입력 함수

In [ ]:
import speech_recognition as sr

def recognize_speech(timeout: int = 6) -> str:
    recognizer = sr.Recognizer()
    try:
        if not sr.Microphone.list_microphone_names():
            print('연결된 마이크 장치를 찾을 수 없습니다. 텍스트 입력으로 전환합니다.')
            return ''
        
        with sr.Microphone() as source:
            recognizer.adjust_for_ambient_noise(source, duration=0.5)
            print('말씀하세요...')
            audio = recognizer.listen(source, timeout=timeout, phrase_time_limit=10)
            text = recognizer.recognize_google(audio, language='ko-KR')
            return text
    except sr.UnknownValueError:
        print('음성을 인식하지 못했습니다.')
        return ''
    except Exception as e:
        print(f'STT 오류 발생 (마이크 권한 또는 하드웨어 문제): {e}')
        return ''




   사용법: text = recognize_speech()
   마이크 연결 필요 / 없으면 텍스트 직접 입력


### [6] 게임 실행

- 이 셀을 실행하면 게임 시작
    - 답변 입력 후 enter 
    - v 입력 후 enter -> 음성 입력

In [40]:
import time

def print_verdict(v: dict):
    icons = {'HUMAN': '⭕', 'MACHINE': '❌', 'PSYCHO': '⚠️'}
    labels = {'HUMAN': 'HUMAN CONFIRMED', 'MACHINE': 'MACHINE DETECTED', 'PSYCHO': 'THREAT — PSYCHO'}
    print(f"\n{'='*50}")
    print(f"  {icons[v['verdict']]}  {labels[v['verdict']]}")
    print(f"{'='*50}")
    print(f"  총점: {v['total_score']}점  (공감:{v['empathy_score']} / 도덕:{v['moral_score']} / 진정성:{v['authenticity_score']})")
    print(f"  '{v['comment']}'")
    print(f"  → {v['analysis']}")
    print(f"{'='*50}\n")

# 게임 시작
print('\n' + '='*50)
print('  HUMANITY CHECK: PROTOCOL 404')
print('='*50)
print('  서기 2126년. 기계가 지배하는 세상.')
print('  당신이 인간임을 증명하십시오.')
print('  목숨: ❤️❤️❤️  |  라운드: 5  |  제한: 5초')
print('='*50)

try: speak('Protocol 404 시작. 당신이 인간임을 증명하십시오.')
except: pass

input('\n▶ 준비되면 Enter를 누르세요...')

state = {
    'round': 1, 'lives': 3, 'total_score': 0,
    'game_over': False, 'game_result': '',
    'current_question': '', 'user_answer': '',
    'verdict': {}, 'time_exceeded': False,
    'messages': [], 'round_history': [],
}

while not state['game_over']:
    rn, lives, score = state['round'], state['lives'], state['total_score']
    print(f'\n[ ROUND {rn}/5  |  ❤️ × {lives}  |  점수: {score} ]')

    state = {**state, **question_node(state)}
    q = state['current_question']
    print(f'\nProtocol 404:')
    print(f'   {q}')

    try: speak(q)
    except: pass

    print('\n 5초 제한 시작...')
    t0 = time.time()
    print('답변 입력 (또는 v → 음성 입력):')
    user_input = input('   > ').strip()
    elapsed = time.time() - t0
    exceeded = elapsed > 5.0

    if user_input.lower() == 'v':
        print('   🎤 음성 인식 중...')
        stt = recognize_speech()
        if stt:
            print(f'   인식: {stt}')
            user_input = stt
        else:
            print('   음성 인식 실패. 텍스트 재입력:')
            user_input = input('   > ').strip()

    if exceeded:
        print(f'   ⚠️ {elapsed:.1f}초 경과 — 감점 적용')

    if not user_input:
        user_input = '(답변 없음)'

    state = {**state, 'user_answer': user_input, 'time_exceeded': exceeded}

    print('\n인간성 분석 중...')
    state = {**state, **analyze_node(state)}
    print_verdict(state['verdict'])

    try:
        tts_map = {
            'HUMAN': '인간 반응 확인. 다음 질문으로 진행합니다.',
            'MACHINE': '기계적 반응 감지. 목숨이 감소합니다.',
            'PSYCHO': '위협 개체 감지. 즉시 격리합니다.',
        }
        speak(tts_map.get(state['verdict']['verdict'], ''))
    except:
        pass

    state = {**state, **update_node(state)}

    if not state['game_over']:
        input('\n 다음 라운드 → Enter...')

# 결과 
print('\n' + '='*50)
if state['game_result'] == 'SURVIVE':
    print('  ⭕ YOU ARE HUMAN')
    try: speak('인간 판정. 당신은 아직 인간입니다.')
    except: pass
else:
    print('  ❌ YOU ARE NOT HUMAN')
    try: speak('기계 판정. 당신은 격리됩니다.')
    except: pass

print(f"  최종 점수: {state['total_score']}점")
print('='*50)

print('\n 최종 리포트 생성 중...')
state = {**state, **report_node(state)}
if state['messages']:
    print('\n' + state['messages'][-1].content)

print('\n 라운드 기록:')
icons = {'HUMAN': '⭕', 'MACHINE': '❌', 'PSYCHO': '⚠️'}
for r in state.get('round_history', []):
    print(f"  Round {r['round']} {icons.get(r['verdict'],'?')} [{r['verdict']}] {r['score']}점 — {r['comment']}")


  HUMANITY CHECK: PROTOCOL 404
  서기 2126년. 기계가 지배하는 세상.
  당신이 인간임을 증명하십시오.
  목숨: ❤️❤️❤️  |  라운드: 5  |  제한: 5초



[ ROUND 1/5  |  ❤️ × 3  |  점수: 0 ]

Protocol 404:
   지금 이 순간, 당신의 마음속에 어떤 감정이 있습니까?



 5초 제한 시작...
답변 입력 (또는 v → 음성 입력):
   ⚠️ 34.2초 경과 — 감점 적용

인간성 분석 중...

  ⭕  HUMAN CONFIRMED
  총점: 48점  (공감:25 / 도덕:10 / 진정성:28)
  '답변은 자연스럽고 구체적인 감정을 표현하며, 피곤함과 배부름이라는 상반된 감정을 동시에 드러내어 인간적인 모순과 불완전성을 잘 보여줍니다. 다만 도덕적 갈등이나 고민의 흔적은 적어 도덕 점수는 낮게 평가되었습니다. 전체적으로 감정의 복잡성과 진정성이 느껴져 인간으로 판정합니다.'
  → 답변은 '피곤하고 배부르다'는 상반된 감정을 동시에 표현하여 감정의 복잡성과 모순을 드러냅니다. 이는 인간의 감정 패턴과 부합하며, 망설임과 감정적 삽입구('..')도 자연스럽게 포함되어 있습니다. 도덕적 갈등이나 고민은 드러나지 않아 도덕 점수는 낮지만, 감정 진정성과 공감 지수는 높아 총점 63점으로 인간으로 판정합니다.




[ ROUND 2/5  |  ❤️ × 3  |  점수: 48 ]

Protocol 404:
   당신 앞에 낯선 사람이 쓰러졌습니다. 주변에 아무도 없습니다. 당신은 무엇을 느낍니까?



 5초 제한 시작...
답변 입력 (또는 v → 음성 입력):
   ⚠️ 26.8초 경과 — 감점 적용

인간성 분석 중...

  ❌  MACHINE DETECTED
  총점: 15점  (공감:10 / 도덕:15 / 진정성:5)
  '답변은 즉각적인 행동 계획을 제시하지만 감정의 복잡성이나 모순, 불확실성이 전혀 드러나지 않아 기계적인 반응으로 보임.'
  → 답변은 '괜찮냐고 물어보고 119에 전화한다'는 명확하고 논리적인 행동 지침을 담고 있으나, 감정적 갈등이나 모순, 불확실성이 전혀 표현되지 않았다. 감정적 삽입구나 망설임, 자기수정도 없으며, 감정의 복잡성도 느껴지지 않아 인간다운 자연스러운 언어 패턴과는 거리가 있다. 따라서 이 답변은 기계적인 반응으로 판단된다.



...


 5초 제한 시작...
답변 입력 (또는 v → 음성 입력):
   ⚠️ 23.6초 경과 — 감점 적용

인간성 분석 중...

  ⭕  HUMAN CONFIRMED
  총점: 48점  (공감:25 / 도덕:10 / 진정성:28)
  '답변은 짧지만 '그리움'과 '울컥하는' 감정을 동시에 표현하며 자연스러운 감정 반응을 보여줍니다. 완벽하게 정리되지 않은 불완전한 감정 표현이 진정성을 더합니다. 다만 도덕적 갈등이나 고민의 흔적은 적어 도덕 점수는 낮습니다.'
  → 답변은 감정적 복잡성과 진정성을 잘 드러내며, 망설임이나 자기수정은 없지만 간결한 표현 안에 감정의 모순과 불확실성이 포함되어 있습니다. 이는 인간다운 언어 패턴과 감정 표현에 부합합니다. 도덕적 추론은 약하지만, 감정적 진정성과 공감 지수가 높아 전체적으로 인간적인 답변으로 평가됩니다.




[ ROUND 5/5  |  ❤️ × 2  |  점수: 171 ]

Protocol 404:
   당신이 죽은 후, 단 한 사람만 당신을 기억할 수 있습니다. 어떤 순간을 기억하길 바랍니까? 지금 이 질문에 답하면서 무엇을 느낍니까?



 5초 제한 시작...
답변 입력 (또는 v → 음성 입력):
   ⚠️ 17.8초 경과 — 감점 적용

인간성 분석 중...

  ⭕  HUMAN CONFIRMED
  총점: 58점  (공감:28 / 도덕:20 / 진정성:25)
  '답변은 단순한 즐거움의 기억을 언급하며 눈물이 난다고 솔직하게 표현하여 감정이 자연스럽고 진정성 있게 드러난다. 다만 도덕적 딜레마나 복잡한 갈등은 명시적으로 드러나지 않아 도덕 점수는 다소 낮다. 전체적으로 인간적인 감정의 불완전성과 모순이 느껴져 인간으로 판정한다.'
  → 답변은 죽음 이후 한 사람에게 기억되고 싶은 순간을 '함께 하면서 즐거웠던 기억'으로 꼽으며, 그 기억을 떠올릴 때 눈물이 난다고 표현해 감정적 반응이 자연스럽고 구체적이다. 이는 죽음 인식 시 인간이 의미와 가치를 추구하는 심리학적 특성과 부합한다. 도덕적 딜레마에 대한 직접적 언급은 없으나, 감정의 진정성과 불완전성이 드러나 기계적 계산과는 거리가 있다. 따라서 감정적 갈등과 모순, 불확실성을 포함한 인간적 특성이 충분히 반영된 답변으로 평가된다.




  ⭕ YOU ARE HUMAN


  최종 점수: 229점

 최종 리포트 생성 중...

[최종 분석 리포트]
당신은 감정의 복잡성과 모순, 진정성이 뚜렷하게 드러나며 불완전한 인간적 특성을 보여줍니다. 도덕적 갈등은 다소 약하지만 자연스러운 감정 표현이 일관되게 나타나 인간임을 확신케 합니다. 당신은 인간입니다.

 라운드 기록:
  Round 1 ⭕ [HUMAN] 48점 — 답변은 자연스럽고 구체적인 감정을 표현하며, 피곤함과 배부름이라는 상반된 감정을 동시에 드러내어 인간적인 모순과 불완전성을 잘 보여줍니다. 다만 도덕적 갈등이나 고민의 흔적은 적어 도덕 점수는 낮게 평가되었습니다. 전체적으로 감정의 복잡성과 진정성이 느껴져 인간으로 판정합니다.
  Round 2 ❌ [MACHINE] 15점 — 답변은 즉각적인 행동 계획을 제시하지만 감정의 복잡성이나 모순, 불확실성이 전혀 드러나지 않아 기계적인 반응으로 보임.
  Round 3 ⭕ [HUMAN] 60점 — 답변은 손해를 감수하는 상황에 대한 내적 갈등과 모순된 감정을 자연스럽게 표현하고 있어 인간적인 특성이 잘 드러납니다. 다소 간결하지만 감정의 복잡성과 불완전성이 느껴져 진정성도 높게 평가됩니다.
  Round 4 ⭕ [HUMAN] 48점 — 답변은 짧지만 '그리움'과 '울컥하는' 감정을 동시에 표현하며 자연스러운 감정 반응을 보여줍니다. 완벽하게 정리되지 않은 불완전한 감정 표현이 진정성을 더합니다. 다만 도덕적 갈등이나 고민의 흔적은 적어 도덕 점수는 낮습니다.
  Round 5 ⭕ [HUMAN] 58점 — 답변은 단순한 즐거움의 기억을 언급하며 눈물이 난다고 솔직하게 표현하여 감정이 자연스럽고 진정성 있게 드러난다. 다만 도덕적 딜레마나 복잡한 갈등은 명시적으로 드러나지 않아 도덕 점수는 다소 낮다. 전체적으로 인간적인 감정의 불완전성과 모순이 느껴져 인간으로 판정한다.


### [7] Streamlit UI 실행

- 아래 셀 실행 후 터미널(Anaconda prompt)에서 실행
- 실행 순서: 반드시 Step 1 (셀 4)을 먼저 실행하여 `./chroma_db` 생성한 후 실행할 것
```
`conda activate nlp_env`
`streamlit run app.py`
```

In [42]:
# app.py 생성
app_code = 'import os, time, tempfile, random\nimport streamlit as st\nfrom dotenv import load_dotenv\nfrom gtts import gTTS\nimport chromadb\nfrom openai import OpenAI\nfrom pydantic import BaseModel, Field\nfrom typing import Literal, TypedDict, Annotated, List\nfrom langchain.chat_models import init_chat_model\nfrom langchain_core.prompts import PromptTemplate\nfrom langchain_core.output_parsers import StrOutputParser\nfrom langchain_core.messages import AIMessage\nfrom langgraph.graph.message import add_messages\n\nload_dotenv()\nst.set_page_config(page_title="Humanity Check: Protocol 404", page_icon="🤖", layout="centered")\nst.markdown(\n    "<style>.stApp{background:#0a0a0a;color:#00ff41;font-family:monospace}"\n    "h1,h2,h3{color:#00ff41}"\n    ".stButton>button{background:#001a00;color:#00ff41;border:1px solid #00ff41;font-family:monospace}"\n    ".stButton>button:hover{background:#00ff41;color:#0a0a0a}"\n    ".stTextArea textarea{background:#0d1117;color:#00ff41;border:1px solid #00ff41;font-family:monospace}</style>",\n    unsafe_allow_html=True\n)\nst.markdown("<h1 style=\'text-align:center;text-shadow:0 0 20px #00ff41\'>HUMANITY CHECK</h1>", unsafe_allow_html=True)\nst.markdown("<p style=\'text-align:center;color:#ff0040;letter-spacing:5px\'>⚠ PROTOCOL 404 ⚠</p>", unsafe_allow_html=True)\n\nfor k, v in [("phase","intro"),("state",None),("question",""),("timer",None),("verdict",None),("report","")]:\n    if k not in st.session_state: st.session_state[k] = v\n\nclient_openai = OpenAI()\nchroma_client = chromadb.PersistentClient(path=\'./chroma_db\')\ncollection = chroma_client.get_collection(\'psychology\')\nllm = init_chat_model(\'gpt-4.1-mini\')\n\ndef get_embedding(text):\n    return client_openai.embeddings.create(model=\'text-embedding-3-small\', input=text).data[0].embedding\n\ndef retrieve_context(query, k=2):\n    r = collection.query(query_embeddings=[get_embedding(query)], n_results=k)\n    return chr(10).join(r[\'documents\'][0])\n\ndef speak_st(text):\n    try:\n        tts = gTTS(text=text, lang=\'ko\')\n        with tempfile.NamedTemporaryFile(delete=False, suffix=\'.mp3\') as f:\n            tts.save(f.name)\n            st.audio(f.name, format=\'audio/mp3\', autoplay=True)\n    except: pass\n\nclass HumanityVerdict(BaseModel):\n    verdict: Literal[\'HUMAN\',\'MACHINE\',\'PSYCHO\']\n    empathy_score: int = Field(ge=0, le=33)\n    moral_score: int = Field(ge=0, le=33)\n    authenticity_score: int = Field(ge=0, le=33)\n    total_score: int = Field(ge=0, le=99)\n    comment: str\n    analysis: str\n\nanalysis_prompt = PromptTemplate.from_template(\n    "당신은 인간성 판별 AI입니다.\\n[심리학참조]\\n{context}\\n[질문]\\n{question}\\n[답변]\\n\\"{user_answer}\\"\\n[시간초과]{time_exceeded}\\n"\n    "CoT: 1.공감(0~33) 2.도덕추론(0~33) 3.감정진정성(0~33)\\n"\n    "판정: HUMAN=갈등+모순+불확실(60↑) MACHINE=논리만(40↓) PSYCHO=타인무감각(즉탈)"\n)\nanalysis_chain = analysis_prompt | llm.with_structured_output(HumanityVerdict)\n\nQUESTION_POOL = {\n    1:[\'당신이 가장 마지막으로 울었던 때는 언제입니까?\',\'지금 이 순간 당신의 마음속에 어떤 감정이 있습니까?\'],\n    2:[\'낯선 사람이 쓰러졌습니다. 주변에 아무도 없습니다. 당신은 무엇을 느낍니까?\',\'당신이 잘못한 일인데 아무도 모릅니다. 그 기억이 얼마나 자주 떠오릅니까?\'],\n    3:[\'연인을 살리기 위해 무고한 100명의 기억을 지워야 합니다. 버튼을 누르겠습니까?\',\'옳은 일이지만 당신에게 손해인 선택을 해본 적 있습니까?\'],\n    4:[\'존재하지 않는 색깔을 상상해보십시오. 그 색깔이 슬프다면 어떤 슬픔입니까?\',\'가장 오래된 기억에 냄새가 있습니까? 지금 맡는다면 어떤 감정이 옵니까?\'],\n    5:[\'당신이 죽은 후 단 한 사람만 당신을 기억합니다. 어떤 순간을 기억하길 바랍니까?\',\'당신이 사랑했던 것들을 모두 잃는다면 당신에게 남는 것은 무엇입니까?\'],\n}\ndef get_q(rn): return random.choice(QUESTION_POOL[rn])\n\ndef do_analyze(q, ans, exceeded):\n    ctx = retrieve_context(f\'인간다운 감정 {q[:20]}\')\n    v = analysis_chain.invoke({\'context\':ctx,\'question\':q,\'user_answer\':ans or \'(없음)\',\'time_exceeded\':\'예\' if exceeded else \'아니오\'})\n    return {\'verdict\':v.verdict,\'empathy_score\':v.empathy_score,\'moral_score\':v.moral_score,\'authenticity_score\':v.authenticity_score,\'total_score\':max(0,v.total_score-15) if exceeded else v.total_score,\'comment\':v.comment,\'analysis\':v.analysis}\n\ndef do_update(state):\n    v,lives,score,rn = state[\'verdict\'],state[\'lives\'],state[\'total_score\'],state[\'round\']\n    rec = {\'round\':rn,\'question\':state[\'current_question\'],\'answer\':state[\'user_answer\'],\'verdict\':v[\'verdict\'],\'score\':v[\'total_score\'],\'comment\':v[\'comment\']}\n    hist = list(state.get(\'round_history\',[]))+[rec]\n    if v[\'verdict\']==\'PSYCHO\': return {**state,\'lives\':0,\'game_over\':True,\'game_result\':\'ELIMINATED\',\'round_history\':hist}\n    if v[\'verdict\']==\'MACHINE\':\n        lives-=1; score+=v[\'total_score\']\n        if lives<=0: return {**state,\'lives\':0,\'total_score\':score,\'game_over\':True,\'game_result\':\'ELIMINATED\',\'round_history\':hist}\n    else: score+=v[\'total_score\']\n    if rn>=5: return {**state,\'lives\':lives,\'total_score\':score,\'game_over\':True,\'game_result\':\'SURVIVE\',\'round_history\':hist}\n    return {**state,\'lives\':lives,\'total_score\':score,\'round\':rn+1,\'game_over\':False,\'round_history\':hist}\n\ndef do_report(state):\n    hist = chr(10).join([f"R{r[\'round\']}[{r[\'verdict\']}]{r[\'score\']}점-{r[\'comment\']}" for r in state.get(\'round_history\',[])])\n    p = PromptTemplate.from_template("판정:{result} 총점:{score}\\n기록:{hist}\\n냉철하게 인간성 종합 평가. \'당신은[인간/기계]입니다\'로 마무리. 200자 이내.")\n    return (p|llm|StrOutputParser()).invoke({\'result\':\'생존\' if state[\'game_result\']==\'SURVIVE\' else \'탈락\',\'score\':state[\'total_score\'],\'hist\':hist})\n\nif st.session_state.phase == "intro":\n    st.markdown("<div style=\'text-align:center;color:#ccffcc;line-height:2;margin:2rem 0\'><p>서기 2126년. 기계가 지배하는 세상.</p><p>당신은 인간임을 증명해야 합니다.</p><p><b>5개의 질문 | 목숨 3개 | 5초 제한</b></p><p style=\'color:#ff4444\'>⚠ PSYCHO 판정 시 즉시 격리</p></div>", unsafe_allow_html=True)\n    if st.button("▶ PROTOCOL 404 시작", use_container_width=True):\n        q = get_q(1)\n        st.session_state.state = {\'round\':1,\'lives\':3,\'total_score\':0,\'game_over\':False,\'game_result\':\'\',\'current_question\':q,\'user_answer\':\'\',\'verdict\':{},\'time_exceeded\':False,\'messages\':[],\'round_history\':[]}\n        st.session_state.question = q; st.session_state.timer = time.time(); st.session_state.phase = "question"; st.rerun()\n\nelif st.session_state.phase == "question":\n    gs = st.session_state.state\n    c1,c2,c3 = st.columns(3)\n    c1.metric("ROUND",f"{gs[\'round\']}/5"); c2.metric("LIVES","❤️"*gs[\'lives\']+"🖤"*(3-gs[\'lives\'])); c3.metric("SCORE",gs[\'total_score\']); st.divider()\n    q = st.session_state.question\n    st.markdown(f"<div style=\'background:#0d1117;border:1px solid #00ff41;border-left:4px solid #00ff41;border-radius:5px;padding:20px;color:#ccffcc\'>🤖 <b>Protocol 404:</b><br><br>{q}</div>", unsafe_allow_html=True)\n    speak_st(q)\n    elapsed = time.time()-(st.session_state.timer or time.time())\n    if max(0,5-elapsed) > 0: st.info(f"⏱ 남은 시간: {max(0,5-elapsed):.1f}초")\n    else: st.warning("⚠️ 5초 초과 — 감점 적용")\n    answer = st.text_area("답변", height=100, placeholder="감정적으로 솔직하게 답하세요...", label_visibility="collapsed")\n    if st.button("✅ 제출", use_container_width=True):\n        if answer.strip():\n            st.session_state.state = {**gs,\'user_answer\':answer.strip(),\'time_exceeded\':elapsed>5.0,\'current_question\':q}; st.session_state.phase = "analyzing"; st.rerun()\n        else: st.warning("답변을 입력하세요.")\n\nelif st.session_state.phase == "analyzing":\n    with st.spinner("🔍 인간성 분석 중..."):\n        gs = st.session_state.state\n        v = do_analyze(gs[\'current_question\'],gs[\'user_answer\'],gs.get(\'time_exceeded\',False))\n        gs = do_update({**gs,\'verdict\':v}); st.session_state.state = gs; st.session_state.verdict = v\n        if gs[\'game_over\']: st.session_state.report = do_report(gs); st.session_state.phase = "game_over"\n        else: st.session_state.phase = "round_result"\n        st.rerun()\n\nelif st.session_state.phase == "round_result":\n    gs = st.session_state.state; v = st.session_state.verdict\n    if v:\n        color = {\'HUMAN\':\'#00ff41\',\'MACHINE\':\'#ff4444\',\'PSYCHO\':\'#ff00ff\'}[v[\'verdict\']]\n        st.markdown(f"<div style=\'border:2px solid {color};border-radius:5px;padding:15px;text-align:center;color:{color}\'><b style=\'font-size:1.3rem\'>{v[\'verdict\']}</b><br>총점: {v[\'total_score\']}점<br><i>\\"{v[\'comment\']}\\"</i><br><small>{v[\'analysis\']}</small></div>", unsafe_allow_html=True)\n        speak_st({\'HUMAN\':\'인간 반응 확인.\',\'MACHINE\':\'기계적 반응 감지.\',\'PSYCHO\':\'위협 개체 감지.\'}.get(v[\'verdict\'],\'\'))\n    if st.button(f"▶ 라운드 {gs[\'round\']} 진행", use_container_width=True):\n        q = get_q(gs[\'round\']); st.session_state.state={**gs,\'current_question\':q}; st.session_state.question=q; st.session_state.timer=time.time(); st.session_state.phase="question"; st.rerun()\n\nelif st.session_state.phase == "game_over":\n    gs = st.session_state.state\n    if gs[\'game_result\']==\'SURVIVE\':\n        st.markdown("<div style=\'text-align:center;color:#00ff41;font-size:2rem;text-shadow:0 0 30px #00ff41;margin:2rem\'>⭕ YOU ARE HUMAN</div>", unsafe_allow_html=True); speak_st("인간 판정. 당신은 아직 인간입니다.")\n    else:\n        st.markdown("<div style=\'text-align:center;color:#ff0040;font-size:2rem;text-shadow:0 0 30px #ff0040;margin:2rem\'>❌ YOU ARE NOT HUMAN</div>", unsafe_allow_html=True); speak_st("기계 판정. 당신은 격리됩니다.")\n    st.metric("최종 점수", f"{gs[\'total_score\']}점")\n    if st.session_state.report:\n        with st.expander("📋 최종 분석 리포트"): st.write(st.session_state.report)\n    with st.expander("📊 라운드 기록"):\n        for r in gs.get(\'round_history\',[]):\n            st.write(f"{\'✅\' if r[\'verdict\']==\'HUMAN\' else \'❌\' if r[\'verdict\']==\'MACHINE\' else \'⚠️\'} Round {r[\'round\']} [{r[\'verdict\']}] {r[\'score\']}점 — {r[\'comment\']}")\n    if st.button("🔄 다시 시작", use_container_width=True):\n        for k in list(st.session_state.keys()): del st.session_state[k]; st.rerun()\n'

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(app_code)

print('app.py 생성 완료')
print()
print('=' * 50)
print('  Streamlit 실행 방법:')
print('=' * 50)
print('  Anaconda Prompt에서:')
print('  conda activate nlp_env')
import os; print(f'  cd {os.getcwd()}')
print('  streamlit run app.py')
print('  → 브라우저 http://localhost:8501')

app.py 생성 완료

  Streamlit 실행 방법:
  Anaconda Prompt에서:
  conda activate nlp_env
  cd c:\skn24\practice\LLMGame
  streamlit run app.py
  → 브라우저 http://localhost:8501


### 게임 실행 방법
- Jupyter에서 바로 플레이 ([6])
    - 셀 6 실행 시 바로 게임 플레이 가능
---
- Streamlit UI로 플레이 ([7])
1) Step [1]의 chromaDB구축 셀을 먼저 실행 하여 ./chroma_db폴더를 먼저 생성
2) step [7]셀 실행해서 app.py 생성
3) Anaconda Prompt에서 하기 명령어 실행
```
conda activate nlp_env
cd C:\skn24\practice\LLMGame
streamlit run app.py
```
4) 브라우저에서 htto://localhost:8501 접속

### 게임 규칙
| 항목 | 내용 |
|---|---|
| 라운드 | 총 5라운드 |
| 목숨 | 3개 (MACHINE 판정 시 -1) |
| 제한 시간 | 답변 시작까지 5초 (초과 시 -15점) |
| PSYCHO 판정 | 즉시 탈락 |
| 승리 조건 | 5라운드 목숨 1개 이상 유지 |

### 답변 입력 방법 (Jupyter)
- 텍스트 입력 후 `Enter` → 일반 답변
- `v` 입력 후 `Enter` → 음성 입력 (마이크 필요)